## Skeleton Code

The code below provides a skeleton for the model building & training component of your project. You can add/remove/build on code however you see fit, this is meant as a starting point.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from glob import glob
%matplotlib inline
import matplotlib.pyplot as plt

##Import any other stats/DL/ML packages you may need here. E.g. Keras, scikit-learn, etc.
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dense, Flatten


## Do some early processing of your metadata for easier model training:

In [ ]:
## Below is some helper code to read all of your full image filepaths into a dataframe for easier manipulation
## Load the NIH data to all_xray_df
all_xray_df = pd.read_csv('/data/Data_Entry_2017.csv')
all_image_paths = {os.path.basename(x): x for x in 
                   glob(os.path.join('/data','images*', '*', '*.png'))}
print('Scans found:', len(all_image_paths), ', Total Headers', all_xray_df.shape[0])
all_xray_df['path'] = all_xray_df['Image Index'].map(all_image_paths.get)
all_xray_df.sample(3)

In [ ]:
## Here you may want to create some extra columns in your table with binary indicators of certain diseases 
## rather than working directly with the 'Finding Labels' column

diseases = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']
for disease in diseases:
    all_xray_df[disease] = all_xray_df['Finding Labels'].map(lambda x: 1.0 if disease in x else 0.0)
print(all_xray_df.sample(3))

In [ ]:
## Here we can create a new column called 'pneumonia_class' that will allow us to look at 
## images with or without pneumonia for binary classification

all_xray_df['pneumonia_class'] = all_xray_df['Pneumonia'].map(lambda x: 1.0 if x == 1.0 else 0.0)
print(all_xray_df.sample(3))

## Training and testing data:

In [ ]:
def create_splits(vargs):
    
    ## Create patient-level stratified split to avoid data leakage
    ## This ensures images from the same patient don't leak between train and val
    ## Also ensures class balance in both sets
    
    # Extract patient ID from the Image Index (typically the first part before the underscore)
    all_xray_df['patient_id'] = all_xray_df['Image Index'].str.split('_').str[0]
    
    # Group by patient and get the majority pneumonia class for that patient
    patient_pneumonia = all_xray_df.groupby('patient_id')['pneumonia_class'].apply(
        lambda x: '1' if (x == '1').sum() > (x == '0').sum() else '0'
    ).reset_index()
    patient_pneumonia.columns = ['patient_id', 'patient_pneumonia_class']
    
    # Split at patient level with stratification
    train_patients, val_patients = train_test_split(
        patient_pneumonia,
        test_size=0.2,
        stratify=patient_pneumonia['patient_pneumonia_class'],
        random_state=42
    )
    
    # Assign images to train/val based on patient split
    train_data = all_xray_df[all_xray_df['patient_id'].isin(train_patients['patient_id'])].copy()
    val_data = all_xray_df[all_xray_df['patient_id'].isin(val_patients['patient_id'])].copy()
    
    print(f'Training set: {len(train_data)} images from {len(train_patients)} patients')
    print(f'Validation set: {len(val_data)} images from {len(val_patients)} patients')
    print(f'Training pneumonia prevalence: {(train_data["pneumonia_class"] == "1").sum() / len(train_data) * 100:.1f}%')
    print(f'Validation pneumonia prevalence: {(val_data["pneumonia_class"] == "1").sum() / len(val_data) * 100:.1f}%')

    return train_data, val_data

# Now we can begin our model-building & training

#### First suggestion: perform some image augmentation on your data

In [ ]:
def my_image_augmentation(vargs):
    
    ## recommendation here to implement a package like Keras' ImageDataGenerator
    ## with some of the built-in augmentations 
    
    ## keep an eye out for types of augmentation that are or are not appropriate for medical imaging data
    ## Also keep in mind what sort of augmentation is or is not appropriate for testing vs validation data
    
    ## STAND-OUT SUGGESTION: implement some of my own custom augmentation that's *not*
    ## built into something like a Keras package

    my_idg = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.01,
        zoom_range=[0.9, 1.25],
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode='reflect',
        data_format='channels_last',
        brightness_range=[0.5, 1.5]
    )
    
    return my_idg


def make_train_gen(vargs):
    
    ## Create the actual generators using the output of my_image_augmentation for your training data
    ## Suggestion here to use the flow_from_dataframe library, e.g.:
    
    my_train_idg = my_image_augmentation(vargs)

    train_gen = my_train_idg.flow_from_dataframe(dataframe=train_data, 
                                         directory=None, 
                                         x_col = 'path',
                                         y_col = 'pneumonia_class',
                                         class_mode = 'binary',
                                         target_size = (224, 224), 
                                         batch_size = 32
                                         )

    return train_gen


def make_val_gen(vargs):
    
    my_val_idg = ImageDataGenerator(rescale=1./255)
        
    val_gen = my_val_idg.flow_from_dataframe(dataframe=val_data, 
                                                directory=None, 
                                                x_col = 'path',
                                                y_col = 'pneumonia_class',
                                                class_mode = 'binary',
                                                target_size = (224, 224), 
                                                batch_size = 32
                                                )
    return val_gen

In [ ]:
# Initialize splits and data generators before using them
# Call create_splits to produce train_data and val_data, then build generators
#train_data, val_data = create_splits(None)
#train_gen = make_train_gen(None)
#val_gen = make_val_gen(None)

## Here we can create a new column called 'pneumonia_class' that will allow us to look at 
## images with or without pneumonia for binary classification
## Use string labels ('0'/'1') because ImageDataGenerator.flow_from_dataframe
## with class_mode='binary' expects y_col values as strings.
all_xray_df['pneumonia_class'] = all_xray_df['Pneumonia'].map(lambda x: '1' if x == 1.0 else '0')
print(all_xray_df[['Pneumonia', 'pneumonia_class']].count().head(10))
print('\nExamples with label 1:')
print(all_xray_df.loc[all_xray_df['pneumonia_class'] == '1', ['Pneumonia', 'pneumonia_class']].head(10))
print('\nExamples with label 0:')
print(all_xray_df.loc[all_xray_df['pneumonia_class'] == '0', ['Pneumonia', 'pneumonia_class']].head(10))

In [ ]:
## May want to look at some examples of our augmented training data. 
## This is helpful for understanding the extent to which data is being manipulated prior to training, 
## and can be compared with how the raw data look prior to augmentation


def create_splits(vargs):
    
    ## It's important to consider here how balanced or imbalanced we want each of those sets to be
    ## for the presence of pneumonia
    
    stratify = all_xray_df['pneumonia_class'] if 'pneumonia_class' in all_xray_df.columns else None

    # Default to a stratified split if no argument is provided
    if vargs == 'random':
        train_data, val_data = train_test_split(all_xray_df, test_size=0.2, random_state=42)
    elif stratify is not None:
        train_data, val_data = train_test_split(
            all_xray_df,
            test_size=0.2,
            stratify=stratify,
            random_state=42
        )
    else:
        train_data, val_data = train_test_split(all_xray_df, test_size=0.2, random_state=42)

    return train_data, val_data

In [ ]:
## May want to pull a single large batch of random validation data for testing after each epoch:
# Ensure the split and validation generator exist before sampling data
if 'train_data' not in globals() or 'val_data' not in globals():
    train_data, val_data = create_splits(None)
if 'val_gen' not in globals():
    val_gen = make_val_gen(None)

valX, valY = next(val_gen)

## Build your model: 

Recommendation here to use a pre-trained network downloaded from Keras for fine-tuning

In [ ]:
def load_pretrained_model(vargs):
    
    # Load pre-trained VGG16 model without the top classification layers
    # This gives us the convolutional base trained on ImageNet
    vgg_model = VGG16(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    
    # Freeze the weights of the pre-trained model so we only train the new layers
    vgg_model.trainable = False
    
    return vgg_model

In [ ]:
def build_my_model(vargs):
    
    my_model = Sequential()
    
    # Add the pre-trained VGG16 convolutional base
    my_model.add(load_pretrained_model(vargs))
    
    # Add custom layers for fine-tuning on our specific task
    my_model.add(Flatten())
    my_model.add(Dense(128, activation='relu'))
    my_model.add(Dense(64, activation='relu'))
    my_model.add(Dense(1, activation='sigmoid'))  # Binary classification output
    
    # Compile the model with appropriate loss and metrics
    my_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    my_model.summary()
    
    return my_model

In [ ]:
## Below is some helper code that will allow you to add checkpoints to your model,
## This will save the 'best' version of your model by comparing it to previous epochs of training

## Note that you need to choose which metric to monitor for your model's 'best' performance if using this code. 
## The 'patience' parameter is set to 10, meaning that your model will train for ten epochs without seeing
## improvement before quitting

## Todo

weight_path = "{}_my_model.best.hdf5".format('xray_class')

checkpoint = ModelCheckpoint(
    weight_path,
    monitor='val_accuracy',
    verbose=1,
    save_best_only=True,
    mode='max',
    save_weights_only=True
)

early = EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=10
)

callbacks_list = [checkpoint, early]

### Start training! 

In [ ]:
## train your model

if 'train_gen' not in globals():
    train_gen = make_train_gen(None)
if 'val_gen' not in globals():
    val_gen = make_val_gen(None)
if 'valX' not in globals() or 'valY' not in globals():
    valX, valY = next(val_gen)

if 'my_model' not in globals():
    my_model = build_my_model(None)

history = my_model.fit(
    train_gen,
    validation_data=(valX, valY),
    epochs=1,
    callbacks=callbacks_list)

##### After training for some time, look at the performance of your model by plotting some performance statistics:

Note, these figures will come in handy for your FDA documentation later in the project

In [ ]:
## After training, make some predictions to assess your model's overall performance
## Note that detecting pneumonia is hard even for trained expert radiologists, 
## so there is no need to make the model perfect.
my_model.load_weights(weight_path)
pred_Y = my_model.predict(valX, batch_size = 32, verbose = True)

In [ ]:
def plot_auc(t_y, p_y):
    
    ## Plot ROC curve with AUC metric
    from sklearn.metrics import roc_curve, auc
        
    fpr, tpr, _ = roc_curve(t_y, p_y)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend()
    plt.show()


def plot_precision_recall(t_y, p_y):
    
    ## Plot precision-recall curve
    from sklearn.metrics import precision_recall_curve, average_precision_score
    
    precision, recall, _ = precision_recall_curve(t_y, p_y)
    avg_precision = average_precision_score(t_y, p_y)
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'PR curve (AP = {avg_precision:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_f1_vs_threshold(t_y, p_y):
    
    ## Plot F1 score across different thresholds
    from sklearn.metrics import f1_score
    
    thresholds = np.arange(0.0, 1.01, 0.01)
    f1_scores = []
    
    for threshold in thresholds:
        predictions = (p_y > threshold).astype(int)
        f1 = f1_score(t_y, predictions)
        f1_scores.append(f1)
    
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, f1_scores, linewidth=2)
    plt.scatter(best_threshold, best_f1, color='red', s=100, label=f'Best F1 = {best_f1:.3f} at threshold = {best_threshold:.2f}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score')
    plt.title('F1 Score vs Classification Threshold')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return best_threshold, best_f1


def plot_confusion_matrix(t_y, p_y, threshold=0.5):
    from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
    import seaborn as sns
    
    predictions = (p_y > threshold).astype(int)
    cm = confusion_matrix(t_y, predictions)
    precision = precision_score(t_y, predictions)
    recall = recall_score(t_y, predictions)
    f1 = f1_score(t_y, predictions)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix (threshold={threshold:.2f})\nPrecision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}')
    plt.show()
    
    return precision, recall, f1


def plot_history(history):
    
    ## Plot training history for loss and accuracy across epochs
    df = pd.DataFrame(history.history)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot loss
    axes[0].plot(df.index, df['loss'], label='Training Loss')
    axes[0].plot(df.index, df['val_loss'], label='Validation Loss')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Plot accuracy
    axes[1].plot(df.index, df['accuracy'], label='Training Accuracy')
    axes[1].plot(df.index, df['val_accuracy'], label='Validation Accuracy')
    axes[1].set_title('Model Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
## plot figures

plot_auc(valY, pred_Y)
plot_precision_recall(valY, pred_Y)
best_threshold, best_f1 = plot_f1_vs_threshold(valY, pred_Y)
print(f'\nOptimal threshold based on F1: {best_threshold:.2f} with F1 = {best_f1:.3f}')
precision, recall, f1 = plot_confusion_matrix(valY, pred_Y, threshold=best_threshold)
plot_history(history)

Once you feel you are done training, you'll need to decide the proper classification threshold that optimizes your model's performance for a given metric (e.g. accuracy, F1, precision, etc.  You decide) 

In [ ]:
## Find the threshold that optimizes your model's F1 score
## F1 is often more appropriate than accuracy for imbalanced datasets

if 'pred_Y' in globals() and 'valY' in globals():
    from sklearn.metrics import f1_score, precision_score, recall_score
    
    thresholds = np.arange(0.0, 1.01, 0.01)
    best_threshold = 0.5
    best_f1 = 0.0
    best_precision = 0.0
    best_recall = 0.0

    for threshold in thresholds:
        predictions = (pred_Y > threshold).astype(int)
        f1 = f1_score(valY.flatten(), predictions.flatten())
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision_score(valY.flatten(), predictions.flatten())
            best_recall = recall_score(valY.flatten(), predictions.flatten())

    print(f'\n=== Threshold Optimization Results (based on F1) ===')
    print(f'Best Threshold: {best_threshold:.2f}')
    print(f'Best F1 Score: {best_f1:.3f}')
    print(f'Precision at best threshold: {best_precision:.3f}')
    print(f'Recall at best threshold: {best_recall:.3f}')

In [ ]:
## Let's look at some examples of true vs. predicted with our best model: 

# Todo

fig, m_axs = plt.subplots(10, 10, figsize=(16, 16))
i = 0
for (c_x, c_y, c_ax) in zip(valX[0:100], valY[0:100], m_axs.flatten()):
    c_ax.imshow(c_x[:, :, 0], cmap='bone')
    if c_y == 1:
        if pred_Y[i] > best_threshold:
            c_ax.set_title('1, 1')
        else:
            c_ax.set_title('1, 0')
    else:
        if pred_Y[i] > best_threshold:
            c_ax.set_title('0, 1')
        else:
            c_ax.set_title('0, 0')
    c_ax.axis('off')
    i = i + 1

In [ ]:
## Just save model architecture to a .json:

model_json = my_model.to_json()
with open("my_model.json", "w") as json_file:
    json_file.write(model_json)